---
# Pipeline de ETL — Ingestão e Tratamento 
(Bronze -> Silver)

---

## 1. Visão Geral
Este notebook é responsável pela execução do pipeline de ingestão, sanitização e transformação dos dados de preços de combustíveis da **ANP (Agência Nacional do Petróleo, Gás Natural e Biocombustíveis)**.

O objetivo principal é extrair os dados em seu formato bruto da camada **Bronze** (armazenados em Volumes do Unity Catalog), aplicar regras de qualidade, tipagem e padronização, e persistir o conjunto resultante no formato **Delta Lake** na camada **Silver**.

---

## 2. Arquitetura e Especificação dos Dados

| Parâmetro | Detalhes / Especificação |
| :--- | :--- |
| **Fonte Primária** | Portal de Dados Abertos da ANP (*Série Histórica de Preços de Combustíveis*) |
| **Camada de Origem (Bronze)** | `/Volumes/workspace/mvp-engenharia-dados/base-combustivel/*.csv` |
| **Formato de Origem** | Arquivos `.csv` (Codificação: `ISO-8859-1`, Delimitador: `;`, com cabeçalho) |
| **Escopo Temporal** | Séries semestrais compreendidas entre **2023/1** e **2026/1** |
| **Camada de Destino (Silver)** | Tabela/Volume na Camada Silver (Formato Delta) |

---

## 3. Escopo do Pipeline e Regras de Negócio (Camada Silver)

Para garantir a confiabilidade, integridade estrutural e alto desempenho de consulta analítica, a rotina realiza o seguinte fluxo de tratamento:

1. **Ingestão Consolidada (Padrão de Leitura em Lote):**
   - Leitura unificada de múltiplos arquivos CSV semestrais utilizando caracteres de substituição (*wildcards* `*.csv`) com tratamento explícito de codificação de texto e delimitadores.

2. **Sanitização do Esquema e Convenção de Nomes:**
   - Padronização de todas as colunas para a convenção `snake_case`.
   - Remoção de espaços, hífens, pontuações e substituição de caracteres especiais para compatibilidade nativa com o Spark e o Delta Lake.

3. **Conversão e Tipagem de Dados:**
   - **Campos Monetários:** Conversão das colunas de valores (`Valor de Venda`, `Valor de Compra`) do formato texto com vírgula para tipo numérico `DecimalType(10, 3)` ou `FloatType`.
   - **Campos Temporais:** Tratamento e conversão da coluna `Data da Coleta` para `DateType` no formato padrão ISO (`AAAA-MM-DD`).

4. **Filtragem de Domínio:**
   - Filtragem seletiva mantendo apenas os produtos delimitados para a análise (**Gasolina Comum** e **Etanol Hidratado**).

5. **Qualidade de Dados e Remoção de Duplicidades:**
   - Limpeza de espaços em branco nas extremidades dos textos (*trim*).
   - Remoção de registros duplicados com base nas chaves de negócio (`CNPJ da Revenda`, `Produto`, `Data da Coleta`).
   - Persistência em tabela ou volume no formato **Delta Lake** com compressão otimizada para consultas analíticas.

In [0]:
from pyspark.sql.functions import col, regexp_replace, to_date, upper, trim, year, month, date_format, count, when

In [0]:
# Leitura de todos (*) os arquivos CSV contidos no volume do Unity Catalog
df_base_combustivel_brasil = spark.read \
    .option("header", "true") \
    .option("sep", ";") \
    .option("encoding", "ISO-8859-1") \
    .csv("/Volumes/workspace/mvp-engenharia-dados/base-combustivel/*.csv")

In [0]:
# Leitura da base
display(df_base_combustivel_brasil.limit(10))

Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
SE,SP,SOROCABA,COMPETRO COMERCIO E DISTRIBUICAO DE DERIVADOS DE PETROLEO LTDA,00.003.188/0001-21,RUA HUMBERTO DE CAMPOS,306,null,JARDIM ZULMIRA,18061-000,ETANOL,03/01/2023,"3,39",null,R$ / litro,BRANCA
SE,SP,SOROCABA,COMPETRO COMERCIO E DISTRIBUICAO DE DERIVADOS DE PETROLEO LTDA,00.003.188/0001-21,RUA HUMBERTO DE CAMPOS,306,null,JARDIM ZULMIRA,18061-000,DIESEL S10,03/01/2023,"6,21",null,R$ / litro,BRANCA
SE,SP,SOROCABA,COMPETRO COMERCIO E DISTRIBUICAO DE DERIVADOS DE PETROLEO LTDA,00.003.188/0001-21,RUA HUMBERTO DE CAMPOS,306,null,JARDIM ZULMIRA,18061-000,GASOLINA,03/01/2023,"4,39",null,R$ / litro,BRANCA
NE,AL,ARAPIRACA,COMERCIAL DE COMBUSTIVEIS E LUBRIFICANTES VITAL LTDA.,02.817.655/0001-82,RODOVIA AL 220 - KM 06,S/N,null,PLANALTO,57308-000,ETANOL,02/01/2023,"3,89",null,R$ / litro,VIBRA ENERGIA
NE,AL,ARAPIRACA,COMERCIAL DE COMBUSTIVEIS E LUBRIFICANTES VITAL LTDA.,02.817.655/0001-82,RODOVIA AL 220 - KM 06,S/N,null,PLANALTO,57308-000,DIESEL,02/01/2023,"6,49",null,R$ / litro,VIBRA ENERGIA
NE,AL,ARAPIRACA,COMERCIAL DE COMBUSTIVEIS E LUBRIFICANTES VITAL LTDA.,02.817.655/0001-82,RODOVIA AL 220 - KM 06,S/N,null,PLANALTO,57308-000,GASOLINA ADITIVADA,02/01/2023,"5,14",null,R$ / litro,VIBRA ENERGIA
NE,AL,ARAPIRACA,COMERCIAL DE COMBUSTIVEIS E LUBRIFICANTES VITAL LTDA.,02.817.655/0001-82,RODOVIA AL 220 - KM 06,S/N,null,PLANALTO,57308-000,GASOLINA,02/01/2023,"5,14",null,R$ / litro,VIBRA ENERGIA
NE,AL,ARAPIRACA,AUTO POSTO SAO FRANCISCO LTDA,12.706.529/0001-04,RUA SAO FRANCISCO,274,null,CENTRO,57300-080,ETANOL,02/01/2023,"3,79",null,R$ / litro,VIBRA ENERGIA
NE,AL,ARAPIRACA,AUTO POSTO SAO FRANCISCO LTDA,12.706.529/0001-04,RUA SAO FRANCISCO,274,null,CENTRO,57300-080,GASOLINA,02/01/2023,"5,29",null,R$ / litro,VIBRA ENERGIA
NE,AL,ARAPIRACA,IBN PINTO E SILVA & CIA LTDA,01.242.690/0001-58,RODOVIA AL-110,678,KM 70,CANAFISTULA,57302-802,ETANOL,02/01/2023,"4,19",null,R$ / litro,IPIRANGA


In [0]:
# Avaliação da base - Leitura do Schema
df_base_combustivel_brasil.printSchema()

root
 |-- Regiao - Sigla: string (nullable = true)
 |-- Estado - Sigla: string (nullable = true)
 |-- Municipio: string (nullable = true)
 |-- Revenda: string (nullable = true)
 |-- CNPJ da Revenda: string (nullable = true)
 |-- Nome da Rua: string (nullable = true)
 |-- Numero Rua: string (nullable = true)
 |-- Complemento: string (nullable = true)
 |-- Bairro: string (nullable = true)
 |-- Cep: string (nullable = true)
 |-- Produto: string (nullable = true)
 |-- Data da Coleta: string (nullable = true)
 |-- Valor de Venda: string (nullable = true)
 |-- Valor de Compra: string (nullable = true)
 |-- Unidade de Medida: string (nullable = true)
 |-- Bandeira: string (nullable = true)



In [0]:
# Avaliação da base - Counts
print("CONTAGEM DE LINHAS E COLUNAs")
print(f"Linhas: {df_base_combustivel_brasil.count()}")
print(f"Colunas: {len(df_base_combustivel_brasil.columns)}")

CONTAGEM DE LINHAS E COLUNAs
Linhas: 3038685
Colunas: 16


In [0]:
# Avaliação da base - Valores null 
print("VALORES NULOS POR COLUNA")

# Itera sobre todas as colunas e conta quantos NULLs existem em cada uma
df_nulos = df_base_combustivel_brasil.select([
    count(when(col(c).isNull() | (col(c) == "null") | (col(c) == ""), c)).alias(c)
    for c in df_base_combustivel_brasil.columns
])
display(df_nulos)

VALORES NULOS POR COLUNA


Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
9114,9114,9114,9114,9114,9114,9614,2345255,14489,9114,9114,9114,9114,3038685,9114,9114


In [0]:
# Avaliação da base - Nulls nas colunas 
display(df_base_combustivel_brasil.filter(col("Regiao - Sigla").isNull()).limit(10))

Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


In [0]:
# Avaliação da base - Registros duplicados
print("REGISTROS DUPLICADOS (na base totoal)")
total = df_base_combustivel_brasil.count()
distintos = df_base_combustivel_brasil.distinct().count()
print(f"Total: {total} | Distintos: {distintos} | Duplicados: {total - distintos}")

REGISTROS DUPLICADOS (na base totoal)
Total: 3038685 | Distintos: 3029556 | Duplicados: 9129


In [0]:
# >>> CAMADA SILVER: TRATAMENTO, LIMPEZA E PADRONIZAÇÃO DE DADOS <<< 

df_base_combustivel_brasil_tratada= (
    df_base_combustivel_brasil
    # 1. NORMALIZAÇÃO DE NOMES DE COLUNAS 
    # 2. LIMPEZA DE TEXTO (trim + upper)
    .withColumn("regiao_sigla", upper(trim(col("Regiao - Sigla"))))
    .withColumn("estado_sigla", upper(trim(col("Estado - Sigla"))))
    .withColumn("municipio", upper(trim(col("Municipio"))))
    .withColumn("revenda", upper(trim(col("Revenda"))))
    .withColumn("cnpj_revenda", trim(col("CNPJ da Revenda")))
    .withColumn("bairro", upper(trim(col("Bairro"))))
    .withColumn("produto", upper(trim(col("Produto"))))
    .withColumn("unidade_medida", trim(col("Unidade de Medida")))
    .withColumn("bandeira", upper(trim(col("Bandeira"))))
    
    # 3. CONVERSÃO DE TIPOS
    # Converte 'Data da Coleta' (dd/MM/yyyy) para DateType
    .withColumn("data_coleta", to_date(col("Data da Coleta"), "dd/MM/yyyy"))
    # Converte 'Valor de Venda' (substitui ',' por '.') para DECIMAL(10,3)
    .withColumn("valor_venda", regexp_replace(col("Valor de Venda"), ",", ".").cast("decimal(10,3)"))
    # Converte 'Valor de Compra' (substitui ',' por '.') para DECIMAL(10,3)
    .withColumn("valor_compra", regexp_replace(col("Valor de Compra"), ",", ".").cast("decimal(10,3)"))
    
    # 4. CRIAÇÃO DE COLUNAS DERIVADAS
    .withColumn("ano_mes", date_format(col("data_coleta"), "yyyy-MM"))
    .withColumn("ano", year(col("data_coleta")))
    .withColumn("mes", month(col("data_coleta")))
    
    # 5. FILTRAGEM DE PRODUTOS DESEJADOS PARA A ANALISE
    # Mantém apenas produtos de interesse ('GASOLINA' e 'ETANOL') e com valor de venda não nulo
    .filter(col("produto").isin("GASOLINA", "ETANOL"))
    .filter(col("valor_venda").isNotNull())
    
    # 6. REMOÇÃO DE COLUNAS IRRELEVANTES
    # Exclui: 'Nome da Rua', 'Numero Rua', 'Complemento' e 'Cep'
    .select(
        "regiao_sigla",
        "estado_sigla",
        "municipio",
        "revenda",
        "cnpj_revenda",
        "bairro",
        "produto",
        "data_coleta",
        "valor_venda",
        "valor_compra",
        "unidade_medida",
        "bandeira",
        "ano_mes",
        "ano",
        "mes"
    )
    
    # 7. DEDUPLICAÇÃO (Remove registros exatamente duplicados)
    .dropDuplicates()
)

In [0]:
# Visualização das alterçãoes na base 

# Leitura da base
display(df_base_combustivel_brasil_tratada.limit(10))

# Avaliação da base - Leitura do Schema
df_base_combustivel_brasil_tratada.printSchema()

# Avaliação da base - Counts
print("\nCONTAGEM DE LINHAS E COLUNAs")
print(f"Linhas: {df_base_combustivel_brasil_tratada.count()}")
print(f"Colunas: {len(df_base_combustivel_brasil_tratada.columns)}")

# Avaliação da base - Valores null 
print("\nVALORES NULOS POR COLUNA")

# Conta os valores nulos (Nulls) em cada coluna
df_nulos = df_base_combustivel_brasil_tratada.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in df_base_combustivel_brasil_tratada.columns
])

display(df_nulos.limit(10))

regiao_sigla,estado_sigla,municipio,revenda,cnpj_revenda,bairro,produto,data_coleta,valor_venda,valor_compra,unidade_medida,bandeira,ano_mes,ano,mes
SE,ES,VILA VELHA,AUTO POSTO E SERVICOS ZUMACH LTDA,02.648.712/0001-47,JARDIM ASTECA,GASOLINA,2023-01-03,4.890,null,R$ / litro,BRANCA,2023-01,2023,1
SE,MG,GOVERNADOR VALADARES,NORTE GV COMBUSTIVEIS LTDA,06.289.407/0001-67,SAO PAULO,GASOLINA,2023-01-03,5.250,null,R$ / litro,IPIRANGA,2023-01,2023,1
CO,GO,CATALAO,POSTO AGUIAR LTDA,01.321.371/0001-38,CENTRO,ETANOL,2023-01-04,3.990,null,R$ / litro,BRANCA,2023-01,2023,1
NE,PI,TERESINA,MIRANTE PETROLEO LTDA,03.612.498/0001-31,MORROS,ETANOL,2023-01-02,4.290,null,R$ / litro,IPIRANGA,2023-01,2023,1
SE,RJ,RIO DE JANEIRO,MARQUES'S PETER AUTO POSTO LTDA,05.570.770/0001-93,ENCANTADO,GASOLINA,2023-01-02,5.180,null,R$ / litro,VIBRA ENERGIA,2023-01,2023,1
S,RS,PASSO FUNDO,AUTO POSTO PRESIDENTE LTDA,92.021.245/0001-13,VERA CRUZ,GASOLINA,2023-01-03,4.890,null,R$ / litro,VIBRA ENERGIA,2023-01,2023,1
SE,MG,CONTAGEM,RIO GRANDE COMBUSTIVEIS LTDA,03.754.139/0001-19,ELDORADO,ETANOL,2023-01-02,4.790,null,R$ / litro,IPIRANGA,2023-01,2023,1
NE,PE,OLINDA,POSTO CAPRI COMERCIO DE PETROLEO LTDA,07.733.200/0002-83,BULTRINS,ETANOL,2023-01-04,3.990,null,R$ / litro,TDC DISTRIBUIDORA,2023-01,2023,1
NE,CE,SOBRAL,V.C.EMPREENDIMENTOS LTDA,03.551.935/0001-54,DOM EXPEDITO,ETANOL,2023-01-04,4.990,null,R$ / litro,IPIRANGA,2023-01,2023,1
SE,MG,PATOS DE MINAS,PARQUE AUTO POSTO LTDA,05.385.747/0001-29,ALTO DOS CAICARAS,ETANOL,2023-01-03,3.790,null,R$ / litro,BRANCA,2023-01,2023,1


root
 |-- regiao_sigla: string (nullable = true)
 |-- estado_sigla: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- revenda: string (nullable = true)
 |-- cnpj_revenda: string (nullable = true)
 |-- bairro: string (nullable = true)
 |-- produto: string (nullable = true)
 |-- data_coleta: date (nullable = true)
 |-- valor_venda: decimal(10,3) (nullable = true)
 |-- valor_compra: decimal(10,3) (nullable = true)
 |-- unidade_medida: string (nullable = true)
 |-- bandeira: string (nullable = true)
 |-- ano_mes: string (nullable = true)
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)


CONTAGEM DE LINHAS E COLUNAs
Linhas: 1455246
Colunas: 15

VALORES NULOS POR COLUNA


regiao_sigla,estado_sigla,municipio,revenda,cnpj_revenda,bairro,produto,data_coleta,valor_venda,valor_compra,unidade_medida,bandeira,ano_mes,ano,mes
0,0,0,0,0,2618,0,0,0,1455246,0,0,0,0,0


In [0]:
# Definindo o nome completo da tabela no Unity Catalog (Catalog.Schema.Tabela)
nome_tabela_gold = "workspace.`mvp-engenharia-dados`.base_combustivel_silver"

# Salvando o DataFrame como Tabela Delta no Unity Catalog
df_base_combustivel_brasil_tratada.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(nome_tabela_gold)

print("Tabela 'base_combustivel_silver' salva com sucesso no Unity Catalog!")

Tabela 'base_combustivel_silver' salva com sucesso no Unity Catalog!
